# Limpieza del corpus

Unifica los diez archivos de reseñas, descarta las calificaciones sin texto y los duplicados, y anonimiza el corpus.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Librerías

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import glob
import nltk
import sklearn
from sklearn.cluster import KMeans

### Verificación del entorno

In [ ]:
# verificar el entorno
print(f"version de pd {pd.__version__}")
print(f"version de sklearn {sklearn.__version__}")

# descargar stopwords básicos de NLTK para probar conexión
nltk.download('stopwords', quiet=True)
print("nltk ok")

### Unificación de los archivos (Sección 3.1)

In [ ]:
# unificar todos los comentarios en un solo dataframe
path = '../csv_parques/' 
all_files = glob.glob(os.path.join(path, "*.csv")) # busca todos los csv en dir, permite q agrega mas csv a futuro
print(f" hay {len(all_files)} csv de comentarios")

lista_dataframes = []

for filename in all_files:
    try:
        # lee el csv
        df_temp = pd.read_csv(filename, index_col=None, header=0)
        
        # columna con nombre del parque para no perder trazabilidad
        nombre_parque = os.path.basename(filename).replace('.csv', '')
        df_temp['nombre_parque'] = nombre_parque
        
        lista_dataframes.append(df_temp)
        print(f"cargado {nombre_parque} | tamanio: {df_temp.shape}")
        
    except Exception as e:
        print(f"error con {filename}: {e}")

if lista_dataframes:
    df = pd.concat(lista_dataframes, axis=0, ignore_index=True) # uso concat porque es mas eficiente que append en loop
    print("exito")
    print(f"total de reviews cargadas: {len(df)}")
    
    print("estructura")
    print(df.info())
    
    # chequeo columnas duplicadas
    print("5 parques al azar ")
    display(df.sample(5))
else:
    print("error, ver que pasa adentro de los csv")

### Limpieza (Sección 3.1)

In [ ]:
# limpieza 
review_column = 'text'
rating_column = 'stars'

df_cleaned = df.dropna(subset=[review_column]).copy() # elimina filas con texto nulo
print(f'reviews despues de eliminar NaNs: {len(df_cleaned)}')

df_cleaned = df_cleaned[df_cleaned[review_column].str.strip().astype(bool)] # elimina filas con texto vacio o solo espacios, que no es lo mismo que nulo
print(f'reviews despues de eliminar textos vacios: {len(df_cleaned)}')

subset_duplicados = ['name', review_column, 'nombre_parque'] 

if 'name' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop_duplicates(subset=subset_duplicados)
else: 
    df_cleaned = df_cleaned.drop_duplicates(subset=[review_column, 'nombre_parque'])

print(f'reviews despues de eliminar duplicados: {len(df_cleaned)}')

cols_to_keep = ['name', review_column, rating_column, 'nombre_parque'] # me quedo con estas columnas
df_cleaned = df_cleaned[cols_to_keep].reset_index(drop=True)  # resetea el índice

df_generic_cleaned_with_name = df_cleaned[cols_to_keep]

print("columnas finales:")
display(df_generic_cleaned_with_name.head())
print(df_generic_cleaned_with_name.info())

### Anonimización

In [ ]:
# df anonimizado
df_generic_cleaned = df_generic_cleaned_with_name.copy()
df_generic_cleaned.insert(0, 'id', range(1, len(df_generic_cleaned) + 1))
df_generic_cleaned = df_generic_cleaned.drop(columns=['name'])

print("columnas finales:")
display(df_generic_cleaned.head())
print(df_generic_cleaned.info())

### Guardado

In [ ]:
df_generic_cleaned.to_pickle("../df_generic_cleaned.pkl")
df_generic_cleaned_with_name.to_pickle("../df_generic_cleaned_with_name.pkl")

### Control del archivo guardado

In [ ]:
import pandas as pd
df = pd.read_pickle("../df_generic_cleaned.pkl")
print(df.columns.tolist())
print(df.head(3))
print(f"\nTotal: {len(df)} | id de {df['id'].min()} a {df['id'].max()}")